In [ ]:
# 04 · Flujo con problem_user (bugs conocidos) · saucedemo (PROD)
#
# Recorre catálogo -> carrito -> checkout con problem_user y deja evidencia de sus bugs.
# Cada validación queda en el log como [OK] o [FALLO] y la ejecución sigue con la siguiente.
# Con problem_user se esperan varios [FALLO]: son bugs que saucedemo trae a propósito.
#
# Imágenes: primero se guardan las imágenes de cada producto con standard_user como referencia.
# Con problem_user cada producto debería mostrar su misma imagen; el bug es que todos muestran sl-404.
#
# Ejecuta la celda. Debajo aparece el log con fecha (dd-mm-aaaa), hora y ambiente.
# El log se guarda en reports/logs/<AMBIENTE>/<aaaa>/<mm>/<dd>/ y las capturas en reports/evidencias/.
# Chrome se abre en modo visible (para ocultarlo: variable de entorno HEADLESS=1).

import contextlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # permite importar pages/ y utils/

from pages import InventoryPage, LoginPage
from utils.config import AMBIENTE, BASE_URL, USERS
from utils.driver_factory import create_driver
from utils.evidence import save_screenshot
from utils.logger import get_logger, setup_execution_log

log_file = setup_execution_log("04_flujo_problem_user")
log = get_logger()
log.info("===== INICIO | Flujo con problem_user =====")
log.info("Ambiente: %s | Sitio: %s", AMBIENTE, BASE_URL)

COMPRADOR = ("Ricardo", "Pavez", "8320000")
fallos = []


def validar(nombre, ok, detalle):
    """Registra [OK]/[FALLO] y sigue; los fallos se revisan al final."""
    if ok:
        log.info("[OK] %s -> %s", nombre, detalle)
    else:
        fallos.append(nombre)
        log.error("[FALLO] %s -> %s", nombre, detalle)


def archivo(src):
    return src.rsplit("/", 1)[-1] if src else "sin imagen"


driver = create_driver()
try:
    # 1. Referencia: imagen de cada producto con standard_user
    inventory = LoginPage(driver).load().login_as(USERS["standard"])
    imagenes_esperadas = inventory.product_images()
    log.info("Imágenes de referencia (standard_user): %d productos", len(imagenes_esperadas))
    inventory.logout()

    # 2. Login con problem_user
    inventory = LoginPage(driver).load().login_as(USERS["problem"])
    log.info("[OK] Login exitoso como %s", USERS["problem"])

    # 3. Imágenes del catálogo
    imagenes = inventory.product_images()
    for producto, esperada in imagenes_esperadas.items():
        actual = imagenes.get(producto)
        validar(f"Imagen de '{producto}'", actual == esperada,
                f"muestra {archivo(actual)} (esperada {archivo(esperada)})")
    distintas = set(imagenes.values())
    validar("Imágenes distintas por producto", len(distintas) == len(imagenes),
            f"{len(imagenes)} productos con {len(distintas)} imagen(es) distinta(s)")
    save_screenshot(driver, "04_imagenes")

    # 4. Orden por precio (menor a mayor)
    inventory.sort_by(InventoryPage.SORT_PRICE_ASC)
    precios = inventory.product_prices()
    validar("Orden por precio", precios == sorted(precios), f"precios en pantalla {precios}")

    # 5. Agregar todos los productos al carrito
    productos = inventory.product_names()
    inventory.add_to_cart(*productos)
    sin_agregar = [p for p in productos if inventory.button_text(p) != "Remove"]
    validar("Agregar al carrito", not sin_agregar,
            f"badge = {inventory.cart_count()} de {len(productos)}; no se agregaron: {', '.join(sin_agregar) or 'ninguno'}")
    save_screenshot(driver, "04_carrito")

    # 6. Datos del comprador en el checkout
    info = inventory.go_to_cart().checkout().fill(*COMPRADOR)
    if info.is_visible(info.ERROR):
        nombre, apellido, codigo = info.field_values()
        validar("Datos del comprador", False,
                f"'{info.error_message()}' | se escribió {COMPRADOR} y quedó nombre='{nombre}' apellido='{apellido}' código='{codigo}'")
    else:
        validar("Datos del comprador", True, "se pasó al resumen de la compra")
    save_screenshot(driver, "04_checkout")

    assert not fallos, f"{len(fallos)} validación(es) fallaron: {', '.join(fallos)}"
    log.info("RESULTADO: todas las validaciones OK")
except Exception:
    log.exception("RESULTADO: la ejecución falló")
    with contextlib.suppress(Exception):
        save_screenshot(driver, "04_error")
    raise
finally:
    driver.quit()
    log.info("===== FIN | Log guardado en %s =====", log_file)